# Clinical Analytics Chatbot
**Drug R&D Assistant — Dataiku Notebook UI**

Run all cells in order. The interactive chat UI launches in the last cell.

**Before running:** set your LLM Mesh connection ID in Cell 2.

`lib/python/` is automatically on the Python path in all Dataiku notebooks — no path manipulation needed.

In [ ]:
# Cell 1 — Initialise Panel (renders interactive widgets inline in the notebook)
import panel as pn
pn.extension('bokeh', sizing_mode='stretch_width')

In [ ]:
# Cell 2 — Configuration
# Replace with your actual Dataiku LLM Mesh connection ID
LLM_CONNECTION_ID = "YOUR_LLM_CONNECTION_ID"

In [ ]:
# Cell 3 — Initialise backend
import yaml
from pathlib import Path
import backend  # resolved from lib/python/backend/ by Dataiku

config_dir = Path(backend.__file__).parent.parent / "config"
with open(config_dir / "llm_config.yaml") as f:
    config = yaml.safe_load(f)
config["llm_mesh"]["connection_id"] = LLM_CONNECTION_ID

from backend.state.session_store import SessionStore
from backend.orchestrator.orchestrator import Orchestrator

session_store = SessionStore(timeout_minutes=60)
orchestrator = Orchestrator(session_store, config=config)
print("Backend initialised.")

In [ ]:
# Cell 4 — Build the interactive UI
import uuid
import pandas as pd
import param
import panel as pn
from panel.chat import ChatInterface, ChatMessage
from backend.state.conversation_state import FSMState
from backend.agents.site_list_merger_agent import parse_uploaded_file


# ── Shared session state ────────────────────────────────────────────────────
SESSION_ID = str(uuid.uuid4())


# ── Helpers ─────────────────────────────────────────────────────────────────
def _make_table(rows, columns):
    """Return a styled Panel DataFrame pane."""
    if not rows:
        return None
    df = pd.DataFrame(rows, columns=columns) if columns else pd.DataFrame(rows)
    return pn.pane.DataFrame(
        df,
        sizing_mode='stretch_width',
        max_rows=200,
        styles={'font-size': '12px'},
    )


def _make_chart(fig):
    """Return a Panel Bokeh pane from a Bokeh Figure object."""
    return pn.pane.Bokeh(fig, sizing_mode='stretch_width')


def _response_to_panel(resp):
    """
    Convert an orchestrator response dict into a Panel Column that can be
    returned from the ChatInterface callback.
    """
    items = []

    if resp.get('message'):
        items.append(pn.pane.Markdown(resp['message'], sizing_mode='stretch_width'))

    if resp.get('table_data'):
        tbl = _make_table(resp['table_data'], resp.get('table_columns'))
        if tbl:
            items.append(tbl)

    if resp.get('chart_json'):
        items.append(_make_chart(resp['chart_json']))

    # Confirmation buttons — injected into the message itself
    if resp.get('fsm_state') == 'confirmation_pending':
        yes_btn = pn.widgets.Button(
            name='✓  Yes, proceed', button_type='success',
            width=160, margin=(8, 6, 4, 0),
        )
        no_btn = pn.widgets.Button(
            name='✗  Cancel', button_type='danger',
            width=110, margin=(8, 0, 4, 0),
        )

        def _confirm(event):
            yes_btn.disabled = True
            no_btn.disabled = True
            r = orchestrator.handle_confirmation(SESSION_ID, confirmed=True)
            chat.send(_response_to_panel(r), user='Assistant', respond=False)
            _maybe_show_export(r)

        def _cancel(event):
            yes_btn.disabled = True
            no_btn.disabled = True
            r = orchestrator.handle_confirmation(SESSION_ID, confirmed=False)
            chat.send(_response_to_panel(r), user='Assistant', respond=False)

        yes_btn.on_click(_confirm)
        no_btn.on_click(_cancel)
        items.append(pn.Row(yes_btn, no_btn, margin=(4, 0, 0, 0)))

    # Export button — shown after a successful skill run
    if resp.get('result_id'):
        _maybe_show_export(resp)

    return pn.Column(*items, sizing_mode='stretch_width') if items else pn.pane.Markdown('...')


# ── Export bar (lives below the chat) ───────────────────────────────────────
_current_result_id = None

export_input  = pn.widgets.TextInput(
    placeholder='Dataiku dataset name…', width=260,
)
export_button = pn.widgets.Button(
    name='⬇  Export to Dataiku Dataset', button_type='primary', width=220,
)
export_status = pn.pane.Markdown('', width=400)
export_row = pn.Row(
    pn.pane.Markdown('**Export last result:**', margin=(8, 8, 0, 0)),
    export_input, export_button, export_status,
    visible=False,
    styles={'background': '#f0f4ff', 'padding': '8px', 'border-radius': '6px'},
)

def _maybe_show_export(resp):
    global _current_result_id
    if resp.get('result_id'):
        _current_result_id = resp['result_id']
        export_row.visible = True

def _on_export(event):
    dataset_name = export_input.value.strip()
    if not dataset_name:
        export_status.object = '⚠ Enter a dataset name first.'
        return
    if not _current_result_id:
        export_status.object = '⚠ No result to export.'
        return
    r = orchestrator.export_to_dataset(SESSION_ID, _current_result_id, dataset_name)
    export_status.object = r.get('message', 'Done.')

export_button.on_click(_on_export)


# ── File upload section ─────────────────────────────────────────────────────
upload_cro     = pn.widgets.FileInput(accept='.csv,.xlsx,.xls', name='CRO Site List')
upload_sponsor = pn.widgets.FileInput(accept='.csv,.xlsx,.xls', name='Sponsor Site List')
upload_status  = pn.pane.Markdown('')


class _FakeFileStorage:
    """Adapts Panel's FileInput bytes to the interface parse_uploaded_file expects."""
    def __init__(self, filename, data):
        self.filename = filename
        self._data = data
    def read(self):
        return self._data


def _handle_upload(file_key, label, widget):
    def _cb(event):
        if widget.value is None:
            return
        filename = widget.filename or 'upload'
        try:
            parsed = parse_uploaded_file(_FakeFileStorage(filename, widget.value))
            state = orchestrator.session_store.get_or_create(SESSION_ID)
            state.uploaded_files[file_key] = parsed
            state.active_skill = 'site_list_merger'
            state.fsm_state = FSMState.PARAMETER_GATHERING
            upload_status.object = (
                f'**{label}:** {filename} — {len(parsed["data"])} rows, '
                f'columns: {", ".join(parsed["columns"])}'
            )
            chat.send(
                f'{label} site list uploaded: **{filename}** '
                f'({len(parsed["data"])} rows)',
                user='Assistant', respond=False,
            )
        except ValueError as e:
            upload_status.object = f'**Error:** {e}'
    return _cb

upload_cro.param.watch(_handle_upload('cro_file', 'CRO', upload_cro), 'value')
upload_sponsor.param.watch(_handle_upload('sponsor_file', 'Sponsor', upload_sponsor), 'value')

upload_section = pn.Column(
    pn.pane.Markdown('#### Site List Files *(for Site List Merger)*'),
    pn.Row(upload_cro, upload_sponsor),
    upload_status,
    styles={'background': '#f9f9f9', 'padding': '10px', 'border-radius': '6px',
            'border': '1px solid #e0e0e0'},
    margin=(0, 0, 8, 0),
)


# ── Main ChatInterface callback ──────────────────────────────────────────────
def chat_callback(contents, user, instance):
    """
    Called when the user sends a message. Returns a Panel object that
    Panel renders as the assistant reply.
    """
    resp = orchestrator.process_message(SESSION_ID, contents)
    panel_resp = _response_to_panel(resp)
    _maybe_show_export(resp)
    return panel_resp


chat = ChatInterface(
    callback=chat_callback,
    user='You',
    avatar='👤',
    callback_user='Assistant',
    show_rerun=False,
    show_undo=False,
    show_copy_icon=False,
    placeholder_text='Describe what you need…',
    sizing_mode='stretch_width',
    height=520,
    styles={'border': '1px solid #e0e0e0', 'border-radius': '8px'},
)

# Send the greeting as the first assistant message
chat.send(
    pn.pane.Markdown(
        "Hello! I'm your **Clinical Analytics Assistant**. I can help you with:\n\n"
        "1. **Site List Merger** — Upload CRO and sponsor site lists to merge them  \n"
        "2. **Trial Benchmarking** — Benchmark trials by indication, age group, and phase  \n"
        "3. **Drug Reimbursement** — Assess reimbursement outlook by country  \n"
        "4. **Enrollment Forecasting** — Generate pessimistic / moderate / optimistic enrollment curves  \n\n"
        "What would you like to do?"
    ),
    user='Assistant', respond=False,
)

print("UI built — run the next cell to display it.")

In [ ]:
# Cell 5 — Launch the interactive UI
app = pn.Column(
    pn.pane.Markdown(
        '# Clinical Analytics Assistant\n*Drug R&D | Dataiku Notebook*',
        styles={'background': '#1a1a2e', 'color': 'white',
                'padding': '14px 18px', 'border-radius': '10px 10px 0 0',
                'margin': '0'},
    ),
    upload_section,
    chat,
    export_row,
    sizing_mode='stretch_width',
    styles={'border': '1px solid #ccc', 'border-radius': '10px',
            'overflow': 'hidden', 'max-width': '920px'},
)

app.servable()  # Makes the app available as a Panel server if run with `panel serve`
app             # Displays the UI inline in the Dataiku notebook